## CONNECTING TO FILE_SYSTEM SERVER

In [1]:
! pip install "mcp[cli]" nest_asyncio -q  
# installing MCP python library

In [2]:
# DOING ALL THE NECESSARY IMPORTS

import nest_asyncio
nest_asyncio.apply()

import ollama
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MODEL = "gpt-oss:120b-cloud"   #gemma4:31b-cloud    #qwen3:8b   #gpt-oss:120b-cloud

In [3]:
# CALLING CONNECTING TO THE MCP SERVER [ MCP SERVER WE ARE USING IS:- SERVER-FILESYSTEM MCP SERVER.

async def get_tools():
## just like installing libraries in python here we are telling which server to connect and which directory to give it access.
    params = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", "."]
    )

# When the server runs, it sometimes prints status messages and errors. Rather than letting those clutter your notebook, this line opens a file 
#called server_errors.log and says "send all that noise in here instead." The "w" means "write" (create/overwrite the file). 
    errlog = open("server_errors.log", "w")


# Reads your params and actually launches the server.
# After establishing connection it create two communication channels one for receiving message from server and other for sending message to server.
# we are calling the two methods as read and write.
    async with stdio_client(params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:  # creating session using the read and write tool created above.

# initializing the session , doing the handshake Client and server introduce themselves and agree on how they'll communicate 
# (versions, capabilities, etc.). Nothing useful can happen before this. 
# We await because we're waiting for the server to respond.
            await session.initialize()  
            result = await session.list_tools()  # after handshake get all the tools.
#storing tool
            tools = [
                {"name": t.name,
                 "description": t.description,
                 "input_schema": t.input_schema}
                for t in result.tools
            ]
    errlog.close()
    return tools

tools = await get_tools()

In [4]:
# TOOLS ARE WITH THR HOST NOW.
tools 

[{'name': 'read_file',
  'description': 'Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.',
  'input_schema': {'$schema': 'http://json-schema.org/draft-07/schema#',
   'type': 'object',
   'properties': {'path': {'type': 'string'},
    'tail': {'description': 'If provided, returns only the last N lines of the file',
     'type': 'number'},
    'head': {'description': 'If provided, returns only the first N lines of the file',
     'type': 'number'}},
   'required': ['path']}},
 {'name': 'read_text_file',
  'description': "Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as text regardless of extension. Only works within

In [5]:
# CALLING LLM WHILE SHARING THE TOOLS

message = {"role": "user", "content": "What files are in the current folder?"}
response = ollama.chat(model=MODEL, messages=[message], tools=tools)

In [6]:
response

ChatResponse(model='gpt-oss:120b', created_at='2026-08-14T09:38:18.118177789Z', done=True, done_reason='stop', total_duration=1830101917, load_duration=None, prompt_eval_count=204, prompt_eval_duration=None, eval_count=135, eval_duration=None, message=Message(role='assistant', content='', thinking="We need to list files in current folder. Use function? Possibly there's a tool to list directory. Let's see the available function names: The list shows just type definitions without names. Need to check actual function signatures. Let's request function list? Probably there is a function like list_files, read_file, etc. The names not visible; need to call them. Let's try calling a generic function? Actually the namespace shows many types but not function names—they are blank. This might be formatting issue. Let's attempt to call something like list_files?", images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='list_files', arguments={'path': '.'}))]))

In [7]:
response["message"]["tool_calls"][0]["function"]

Function(name='list_files', arguments={'path': '.'})

In [8]:
# CONNECTING TO SERVER AGAIN AND CALLING THE TOOL

async def call_mcp_tool(tool_name, arguments):
    params = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", "."]
    )
    errlog = open("server_errors.log", "w")
    async with stdio_client(params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)
    errlog.close()
    return result.content[0].text

In [9]:
call = response.message.tool_calls[0].function
output = await call_mcp_tool(call.name, call.arguments)
print(output)  

# NO OUTPUT.

# CHANGE MODEL AND SHOW

MCP error -32602: Tool list_files not found


In [10]:
#ACTUAL TOOL

output = await call_mcp_tool("list_directory", {"path": "."})
print(output)

[DIR] .ipynb_checkpoints
[FILE] server_errors.log
[FILE] Session - 1 MCP Demo.ipynb
[FILE] Session 2 MCP.ipynb


In [11]:
## SOLUTION :- CONVERT TOOLS IN THE FORMAT THAT MCP READS

In [10]:
ollama_tools = [
    {
        "type": "function",
        "function": {
            "name": t["name"],
            "description": t["description"],
            "parameters": t["input_schema"],
        },
    }
    for t in tools
]

In [11]:
ollama_tools

[{'type': 'function',
  'function': {'name': 'read_file',
   'description': 'Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.',
   'parameters': {'$schema': 'http://json-schema.org/draft-07/schema#',
    'type': 'object',
    'properties': {'path': {'type': 'string'},
     'tail': {'description': 'If provided, returns only the last N lines of the file',
      'type': 'number'},
     'head': {'description': 'If provided, returns only the first N lines of the file',
      'type': 'number'}},
    'required': ['path']}}},
 {'type': 'function',
  'function': {'name': 'read_text_file',
   'description': "Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of 

In [12]:
message = {"role": "user", "content": "What files are in the current folder?"}
response = ollama.chat(model=MODEL, messages=[message], tools=ollama_tools)
response

ChatResponse(model='gpt-oss:120b', created_at='2026-08-14T09:43:04.858437518Z', done=True, done_reason='stop', total_duration=1641211795, load_duration=None, prompt_eval_count=1248, prompt_eval_duration=None, eval_count=49, eval_duration=None, message=Message(role='assistant', content='', thinking='We need list files in current folder. Use list_allowed_directories to see root allowed. Then list_directory.', images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='list_directory', arguments={'path': '.'}))]))

In [13]:
print(response.message.tool_calls)

[ToolCall(function=Function(name='list_directory', arguments={'path': '.'}))]


In [14]:
call = response.message.tool_calls[0].function
output = await call_mcp_tool(call.name, call.arguments)
print(output)

[DIR] .ipynb_checkpoints
[FILE] server_errors.log
[FILE] Session - 1 MCP Demo.ipynb
[FILE] Session 2 MCP.ipynb


In [15]:
# TESTING FOR OTHER PROMPTS 
# TEST 1
message = {"role": "user", "content": "Read the contents of the file server_errors.log"}
response = ollama.chat(model="qwen3:8b", messages=[message], tools=ollama_tools)
print(response.message.tool_calls)

[ToolCall(function=Function(name='read_text_file', arguments={'path': 'server_errors.log'}))]


In [16]:
response   # number of tokens

ChatResponse(model='qwen3:8b', created_at='2026-08-14T09:45:30.0841941Z', done=True, done_reason='stop', total_duration=20521638400, load_duration=9977338000, prompt_eval_count=1562, prompt_eval_duration=1703666000, eval_count=166, eval_duration=8814299000, message=Message(role='assistant', content='', thinking="Okay, the user wants to read the contents of the file server_errors.log. Let me check which tool to use. The available functions include read_text_file, which is meant for reading files as text. The read_file function is deprecated, so I should use read_text_file instead. The parameters for read_text_file include the path, and optionally head or tail lines. Since the user didn't specify lines, I'll just provide the path. So the function call should be read_text_file with the path set to server_errors.log. I don't need to include head or tail parameters here. Let me make sure the path is correct and within allowed directories. Alright, that should do it.\n", images=None, tool_na

In [17]:
call = response.message.tool_calls[0].function
output = await call_mcp_tool(call.name, call.arguments)
print(output)

Secure MCP Filesystem Server running on stdio
Client does not support MCP Roots, using allowed directories set from server args: [ 'C:\\Users\\acer\\CampusX\\Context Engineering\\Module 5 MCP' ]



In [18]:
# TEST 2

message = {"role": "user", "content": "Create a new folder called test_folder"}
response = ollama.chat(model=MODEL, messages=[message], tools=ollama_tools)
print(response.message.tool_calls)

call = response.message.tool_calls[0].function
output = await call_mcp_tool(call.name, call.arguments)
print(output)

[ToolCall(function=Function(name='create_directory', arguments={'path': 'test_folder'}))]
Successfully created directory test_folder


In [19]:
response

ChatResponse(model='gpt-oss:120b', created_at='2026-08-14T09:46:42.94033151Z', done=True, done_reason='stop', total_duration=829118358, load_duration=None, prompt_eval_count=1247, prompt_eval_duration=None, eval_count=53, eval_duration=None, message=Message(role='assistant', content='', thinking='We need to create a directory named test_folder. No path given, probably at root of workspace. Use create_directory.', images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='create_directory', arguments={'path': 'test_folder'}))]))

In [21]:
# TEST 3

message = {"role": "user", "content": "Create a file called notes.txt containing the text hello world"}
response = ollama.chat(model="qwen3:8b", messages=[message], tools=ollama_tools)
print(response.message.tool_calls)

call = response.message.tool_calls[0].function
output = await call_mcp_tool(call.name, call.arguments)
print(output)

[ToolCall(function=Function(name='write_file', arguments={'path': 'notes.txt', 'content': 'hello world'}))]
Successfully wrote to notes.txt


In [22]:
# TEST 4

message = {"role": "user", "content": "Rename notes.txt to notes_final.txt"}
response = ollama.chat(model="qwen3:8b", messages=[message], tools=ollama_tools)
print(response.message.tool_calls)

call = response.message.tool_calls[0].function
output = await call_mcp_tool(call.name, call.arguments)
print(output)

[ToolCall(function=Function(name='move_file', arguments={'source': 'notes.txt', 'destination': 'notes_final.txt'}))]
Successfully moved notes.txt to notes_final.txt


## CONNECTING TO MCP-EVERYTHING SERVER

In [20]:
async def explore_everything():
    params = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-everything"]
    )
    errlog = open("server_errors.log", "w")

    async with stdio_client(params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools     = await session.list_tools()
            resources = await session.list_resources()
            prompts   = await session.list_prompts()

            print("=== TOOLS ===")
            for t in tools.tools:
                print(" -", t.name)

            print("\n=== RESOURCES ===")
            for r in resources.resources:
                print(" -", r.uri, "|", r.name)

            print("\n=== PROMPTS ===")
            for p in prompts.prompts:
                print(" -", p.name)

    errlog.close()

await explore_everything()

=== TOOLS ===
 - echo
 - get-annotated-message
 - get-env
 - get-resource-links
 - get-resource-reference
 - get-structured-content
 - get-sum
 - get-tiny-image
 - gzip-file-as-resource
 - toggle-simulated-logging
 - toggle-subscriber-updates
 - trigger-long-running-operation
 - simulate-research-query

=== RESOURCES ===
 - demo://resource/static/document/architecture.md | architecture.md
 - demo://resource/static/document/extension.md | extension.md
 - demo://resource/static/document/features.md | features.md
 - demo://resource/static/document/how-it-works.md | how-it-works.md
 - demo://resource/static/document/instructions.md | instructions.md
 - demo://resource/static/document/startup.md | startup.md
 - demo://resource/static/document/structure.md | structure.md

=== PROMPTS ===
 - simple-prompt
 - args-prompt
 - completable-prompt
 - resource-prompt


In [21]:
# GET A RESOURCE

async def read_resource(uri=None):
    params = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-everything"]
    )
    errlog = open("server_errors.log", "w")
    async with stdio_client(params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # if no uri given, show what's available and stop
            available = await session.list_resources()
            uris = [str(r.uri) for r in available.resources]

            if uri is None:
                errlog.close()
                print("Available resource URIs:")
                for u in uris:
                    print(" -", u)
                return None

            result = await session.read_resource(uri)
            errlog.close()
            return result.contents[0].text

await read_resource()


Available resource URIs:
 - demo://resource/static/document/architecture.md
 - demo://resource/static/document/extension.md
 - demo://resource/static/document/features.md
 - demo://resource/static/document/how-it-works.md
 - demo://resource/static/document/instructions.md
 - demo://resource/static/document/startup.md
 - demo://resource/static/document/structure.md


In [22]:
text = await read_resource("demo://resource/static/document/architecture.md")
print(text)

# Everything Server – Architecture

**Architecture
| [Project Structure](structure.md)
| [Startup Process](startup.md)
| [Server Features](features.md)
| [Extension Points](extension.md)
| [How It Works](how-it-works.md)**

This documentation summarizes the current layout and runtime architecture of the `src/everything` package.
It explains how the server starts, how transports are wired, where tools, prompts, and resources are registered, and how to extend the system.

## High‑level Overview

### Purpose

A minimal, modular MCP server showcasing core Model Context Protocol features. It exposes simple tools, prompts, and resources, and can be run over multiple transports (STDIO, SSE, and Streamable HTTP).

### Design

A small “server factory” constructs the MCP server and registers features.
Transports are separate entry points that create/connect the server and handle network concerns.
Tools, prompts, and resources are organized in their own submodules.

### Multi‑client

The server s

In [24]:
# GET A PROMPT 

async def get_prompt(name=None, arguments=None):
    params = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-everything"]
    )
    errlog = open("server_errors.log", "w")
    async with stdio_client(params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # if no name given, show what's available and stop
            available = await session.list_prompts()
            names = [p.name for p in available.prompts]

            if name is None:
                errlog.close()
                print("Available prompt names:")
                for n in names:
                    print(" -", n)
                return None

            result = await session.get_prompt(name, arguments or {})
            errlog.close()
            return result.messages


await get_prompt()    

Available prompt names:
 - simple-prompt
 - args-prompt
 - completable-prompt
 - resource-prompt


In [25]:
msgs = await get_prompt("simple-prompt")
print(msgs)

[PromptMessage(role='user', content=TextContent(type='text', text='This is a simple prompt without arguments.', annotations=None, meta=None))]


In [ ]:
## FUNCTION TO READ ALL TOOLS [CONVERT IT INTO OLLAMA FORMAT ], READ ALL RESOURCES , READ ALL PROMPTS

In [26]:
import nest_asyncio
nest_asyncio.apply()

import ollama
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MODEL = "gpt-oss:120b-cloud"


async def fetch_mcp_capabilities():
    """Connect to server-everything and return its tools, resources, and prompts."""
    params = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-everything"]
    )
    errlog = open("server_everything_errors.log", "w")

    async with stdio_client(params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result     = await session.list_tools()
            resources_result = await session.list_resources()
            prompts_result   = await session.list_prompts()

            # Tools passed the PROPER way — as function definitions
            ollama_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.input_schema,
                    },
                }
                for t in tools_result.tools
            ]

            # Read the ACTUAL content of every resource (this is what really bloats context)
            resource_dump = "=== RESOURCES ===\n"
            for r in resources_result.resources:
                try:
                    content = await session.read_resource(r.uri)
                    resource_dump += f"--- {r.name} ({r.uri}) ---\n{content.contents[0].text}\n\n"
                except Exception as e:
                    resource_dump += f"[Error reading {r.uri}: {e}]\n\n"

            # Prompt descriptions
            prompt_dump = "=== PROMPTS ===\n"
            for p in prompts_result.prompts:
                desc = getattr(p, "description", "No description")
                prompt_dump += f"{p.name}: {desc}\n"

    errlog.close()
    return ollama_tools, resource_dump, prompt_dump


ollama_tools, resource_dump, prompt_dump = await fetch_mcp_capabilities()
print(f"Fetched {len(ollama_tools)} tools from the server.")

Fetched 13 tools from the server.


In [28]:
ollama_tools

[{'type': 'function',
  'function': {'name': 'echo',
   'description': 'Echoes back the input string',
   'parameters': {'$schema': 'http://json-schema.org/draft-07/schema#',
    'type': 'object',
    'properties': {'message': {'type': 'string',
      'description': 'Message to echo'}},
    'required': ['message']}}},
 {'type': 'function',
  'function': {'name': 'get-annotated-message',
   'description': 'Demonstrates how annotations can be used to provide metadata about content.',
   'parameters': {'$schema': 'http://json-schema.org/draft-07/schema#',
    'type': 'object',
    'properties': {'messageType': {'type': 'string',
      'enum': ['error', 'success', 'debug'],
      'description': 'Type of message to demonstrate different annotation patterns'},
     'includeImage': {'default': False,
      'description': 'Whether to include an example image',
      'type': 'boolean'}},
    'required': ['messageType']}}},
 {'type': 'function',
  'function': {'name': 'get-env',
   'description'

In [29]:
resource_dump

'=== RESOURCES ===\n--- architecture.md (demo://resource/static/document/architecture.md) ---\n# Everything Server – Architecture\n\n**Architecture\n| [Project Structure](structure.md)\n| [Startup Process](startup.md)\n| [Server Features](features.md)\n| [Extension Points](extension.md)\n| [How It Works](how-it-works.md)**\n\nThis documentation summarizes the current layout and runtime architecture of the `src/everything` package.\nIt explains how the server starts, how transports are wired, where tools, prompts, and resources are registered, and how to extend the system.\n\n## High‑level Overview\n\n### Purpose\n\nA minimal, modular MCP server showcasing core Model Context Protocol features. It exposes simple tools, prompts, and resources, and can be run over multiple transports (STDIO, SSE, and Streamable HTTP).\n\n### Design\n\nA small “server factory” constructs the MCP server and registers features.\nTransports are separate entry points that create/connect the server and handle ne

In [30]:
 prompt_dump

'=== PROMPTS ===\nsimple-prompt: A prompt with no arguments\nargs-prompt: A prompt with two arguments, one required and one optional\ncompletable-prompt: First argument choice narrows values for second argument.\nresource-prompt: A prompt that includes an embedded resource reference\n'

In [31]:
def count_input_tokens(messages, tools=None):
    """Send a message to the model and report how many INPUT tokens it consumed."""
    response = ollama.chat(model=MODEL, messages=messages, tools=tools)
    return response.prompt_eval_count


user_msg = [{"role": "user", "content": "HI"}]

# 1. BASELINE — just "HI", nothing attached
baseline = count_input_tokens(user_msg)

# 2. TOOLS ATTACHED — same "HI", but MCP tools passed properly
with_tools = count_input_tokens(user_msg, tools=ollama_tools)

# 3. TOOLS + RESOURCES + PROMPTS — everything MCP can throw at the context
polluted_msg = [
    {"role": "system", "content": resource_dump + "\n" + prompt_dump},
    {"role": "user", "content": "HI"},
]
with_everything = count_input_tokens(polluted_msg, tools=ollama_tools)

print("INPUT TOKENS FOR THE SAME 'HI' MESSAGE")
print("-" * 45)
print(f"1. Plain 'HI' (no MCP)          : {baseline:>6} tokens")
print(f"2. 'HI' + MCP tools             : {with_tools:>6} tokens")
print(f"3. 'HI' + tools + resources     : {with_everything:>6} tokens")

INPUT TOKENS FOR THE SAME 'HI' MESSAGE
---------------------------------------------
1. Plain 'HI' (no MCP)          :     68 tokens
2. 'HI' + MCP tools             :    709 tokens
3. 'HI' + tools + resources     :   8040 tokens
